# 🥑 AguaVerde — Experimento E: Red Neuronal Probabilística
## Clasificación de estrés hídrico por verosimilitud multivariada

| | |
|---|---|
| **Proyecto** | AguaVerde — Cooperativa Aguacatera, Jalisco |
| **Equipo** | Equipo 16 |
| **Experimento** | E — Red Neuronal Probabilística (Transformer + Gaussian NLL) |
| **Complementa** | E3 Stacking (F1=0.8868) + Experimento D (GP por parcela) |
| **Motivación** | Reunión con asesor 2026-06-24: justificar umbrales con likelihood multivariado |

### ¿Qué hace este experimento?

En vez de clasificar estrés con un **umbral fijo** sobre NDMI (que el asesor calificó de
"arbitrario"), entrenamos una red neuronal que aprende la **distribución esperada** de los
5 índices espectrales condicionada a:

1. **Huella espectral de la parcela** — media histórica de cada índice (proxy del tipo de terreno visible vía satélite)
2. **Historial reciente** — últimas 24 observaciones de la serie temporal
3. **Día del año** — estacionalidad

La red predice **(μ, σ) por índice** — los parámetros de una Gaussiana multivariada diagonal.
La clasificación de estrés viene del **percentil** que ocupa la observación en esa distribución
(equivalente a usar la CDF multivariada que explicó el asesor).

```
z_i = (y_obs_i − μ_i) / σ_i            ← cuántas σ está el índice de lo esperado
señal_estrés = −Σ w_i · z_i            ← positivo cuando está POR DEBAJO (estrés)
p = Φ(señal_estrés) × 100              ← percentil bajo la normal estándar

p < 25%  → Sin estrés
p < 60%  → Estrés moderado
p ≥ 60%  → Estrés severo
```


## 1. Setup


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path

# Asegurar que la raíz del proyecto esté en el path
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
import torch
import json

from src.models.likelihood_nn.stress_likelihood_net import (
    StressLikelihoodNet, gaussian_nll,
)
from src.models.likelihood_nn.likelihood_dataset import load_split
from torch.utils.data import DataLoader

# ── Rutas ──────────────────────────────────────────────────────────────
SIGNALS_DIR = ROOT / "data/datasets/signals"
NORM_PATH   = ROOT / "data/datasets/normalizer_stats.json"
SPLIT_JSON  = ROOT / "data/datasets/split.json"
MANIFEST    = ROOT / "data/datasets/manifest.csv"
MODEL_PATH  = ROOT / "models/stress_likelihood_net.pt"

CHANNEL_NAMES = ["NDVI", "NDWI", "NDMI", "NDRE", "EVI"]
WINDOW_SIZE   = 24

# ── Estilos globales de matplotlib ─────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#f0f0ec",
    "grid.linewidth": 0.8,
    "font.size": 10,
})

# Paleta de colores del dashboard
COLOR_GREEN  = "#388E3C"
COLOR_YELLOW = "#F9A825"
COLOR_RED    = "#C62828"
COLOR_GREY   = "#9E9E9E"

print("✅ Setup completo")
print(f"   ROOT:       {ROOT}")
print(f"   MODEL_PATH: {MODEL_PATH}")
print(f"   Modelo entrenado: {MODEL_PATH.exists()}")


## 2. Contexto y motivación

### ¿Por qué un nuevo experimento?

El asesor planteó en la reunión del 24 de junio de 2026 que el camino correcto para
**fundamentar matemáticamente los umbrales de estrés** es estimar la distribución
conjunta de los índices espectrales y clasificar por percentil:

> *"Sería crear una red neuronal que mapee características visuales de la imagen
> satelital con sus diez canales que te entrega Sentinel, más los índices, más la fecha.
> Esas tres cosas sería mapearlas a un likelihood multivariado — un mu y un sigma."*

| Experimento | Clasificación | Umbrales | Datos |
|---|---|---|---|
| **E3 Stacking** | argmax(proba) | percentil global fijo de NDMI | 35 features tabulares |
| **Exp. D — GP** | z-score vs GP del historial | adaptativos por parcela y fecha | serie temporal 1 índice |
| **Exp. E — Likelihood NN** | percentil CDF multivariada | emergen de la distribución aprendida | 5 índices + fecha + terreno |

### Diferencia clave respecto al GP (Experimento D)

- El GP aprende la distribución de **un solo índice** (NDMI) usando el historial de **esa parcela**.
- La red neuronal aprende la distribución de los **5 índices simultáneamente**, condicionada
  a la **huella espectral del terreno** (no solo al historial propio) — generaliza mejor a
  parcelas con pocas observaciones.


## 3. Dataset: ventanas deslizantes

Cada muestra del dataset es una **ventana** de T=24 fechas consecutivas extraída
de los archivos `data/datasets/signals/*.npz`:

```
x_hist   : (24, 5)  — historial normalizado de índices (entrada)
x_static : (5,)    — media histórica completa de la parcela (huella del terreno)
doy      : scalar  — día del año de la fecha objetivo [1, 365]
y_target : (5,)    — valores reales en la fecha objetivo (lo que el modelo predice)
```

La separación **train/val/test** se hace por parcela (no por ventana) para evitar data-leakage.


In [ ]:
with open(SPLIT_JSON) as f:
    split = json.load(f)

train_ds, val_ds, test_ds = load_split(SIGNALS_DIR, SPLIT_JSON, WINDOW_SIZE)

print("─" * 50)
print(f"  Parcelas train : {len(split['train']):3d}  →  {len(train_ds):6,} ventanas")
print(f"  Parcelas val   : {len(split['val']):3d}  →  {len(val_ds):6,} ventanas")
print(f"  Parcelas test  : {len(split['test']):3d}  →  {len(test_ds):6,} ventanas")
print("─" * 50)
print(f"  Total muestras : {len(train_ds)+len(val_ds)+len(test_ds):,}")
print()
print("Muestra [0] del dataset de entrenamiento:")
x_hist_ex, x_static_ex, doy_ex, y_ex = train_ds[0]
print(f"  x_hist   shape: {tuple(x_hist_ex.shape)}")
print(f"  x_static shape: {tuple(x_static_ex.shape)}  →  {x_static_ex.numpy().round(4)}")
print(f"  doy:            {doy_ex.item():.0f}")
print(f"  y_target:       {y_ex.numpy().round(4)}")


In [ ]:
# Carga la serie completa de una parcela de ejemplo y muestra una ventana
PARCEL_VIZ = "H16"
npz_viz = np.load(SIGNALS_DIR / f"{PARCEL_VIZ}.npz", allow_pickle=True)
data_viz = npz_viz["data"].astype(np.float32)    # (T, 5)
dates_viz = npz_viz["dates"]

fig, ax = plt.subplots(figsize=(11, 3.5))
t_range = range(200, 277)
ax.plot(list(t_range), data_viz[200:, 2], color=COLOR_GREEN, lw=1.5, label="NDMI observado")

# Sombrea la última ventana de entrada y la observación objetivo
ax.axvspan(277-WINDOW_SIZE-1, 277-2, alpha=.14, color="royalblue", label=f"Ventana entrada (T={WINDOW_SIZE})")
ax.axvline(276, color=COLOR_RED, lw=2, linestyle="--", label="Fecha objetivo (y)")
ax.set_xlabel("Índice temporal (últimas 77 fechas)")
ax.set_ylabel("NDMI normalizado")
ax.set_title(f"Parcela {PARCEL_VIZ} — ventana de entrada → predicción de distribución")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


## 4. Arquitectura del StressLikelihoodNet

```
┌──────────────────────────────────────────────────────────────────────┐
│  Entradas                                                            │
│                                                                      │
│  x_hist (T=24, 5) ──► Linear(5→64) + PosEmbed ──► Transformer(2L) ─►┐
│                                                     (token final)   │ │
│  x_static (5,) ────► Linear(5→32) + ReLU ─────────────────────────►│ │
│                                                                      │ │
│  doy (scalar) ─────► sin/cos(anual+mensual) → Linear(4→32) + ReLU ►│ │
│                                                                      │ │
│  Fusión: concat(64+32+32) = 128 ──► Linear(128→64) → ReLU          │ │
│                                                                      │ │
│  Cabezas de salida:                                                  │ │
│    mu_head:        Linear(64→5)  → μ (sin activación)               │ │
│    logsigma_head:  Linear(64→5)  → softplus + ε → σ (siempre > 0)  │ │
│                                                                      │ │
│  Pérdida: Gaussian NLL = log(σ) + (y−μ)²/(2σ²)  ← max. verosimilitud│
└──────────────────────────────────────────────────────────────────────┘
```

**¿Por qué Gaussian NLL?** Minimizarla es equivalente a ajustar μ y σ por **máxima
verosimilitud** — exactamente la técnica que sugirió el asesor para fundamentar los umbrales.
Con un umbral fijo estábamos evaluando sobre una CDF, pero sin saber si la PDF subyacente
estaba bien ajustada; este modelo aprende los parámetros de esa PDF directamente.


In [ ]:
if not MODEL_PATH.exists():
    print("⚠️  Modelo no encontrado. Ejecuta primero:")
    print("     python scripts/train_save_likelihood_nn.py")
    print("     (o:  make train-likelihood)")
else:
    ckpt  = torch.load(MODEL_PATH, map_location="cpu", weights_only=False)
    cfg   = ckpt["config"]
    model = StressLikelihoodNet(**cfg)
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Modelo cargado correctamente")
    print(f"  Config:             {cfg}")
    print(f"  Parámetros:         {n_params:,}")
    print(f"  Mejor val NLL:      {ckpt['best_val_nll']:.4f}")
    print()
    print(model)


## 5. Entrenamiento

El modelo ya fue entrenado con `make train-likelihood` (60 épocas, AdamW con
cosine annealing, batch=256). Si quieres reentrenar desde cero:

```bash
python scripts/train_save_likelihood_nn.py --epochs 60
```

La celda siguiente muestra cómo se vería la curva de pérdida si corrieras el entrenamiento
directamente aquí (modo demostración rápida con pocas épocas).


In [ ]:
# Entrenamiento rápido de demostración (10 épocas) para ver la curva de pérdida.
# Para producción usa train_save_likelihood_nn.py (60 épocas).

DEMO_EPOCHS = 10

model_demo = StressLikelihoodNet(n_indices=5, seq_len=WINDOW_SIZE, d_model=64, n_heads=4, n_layers=2)
opt  = torch.optim.AdamW(model_demo.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=DEMO_EPOCHS, eta_min=5e-5)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=256, shuffle=False, num_workers=0)

train_hist, val_hist = [], []
for epoch in range(1, DEMO_EPOCHS + 1):
    model_demo.train()
    tl = []
    for xh, xs, d, y in train_loader:
        mu_b, sig_b = model_demo(xh, xs, d)
        loss = gaussian_nll(mu_b, sig_b, y)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model_demo.parameters(), 1.0)
        opt.step()
        tl.append(loss.item())
    model_demo.eval()
    vl = []
    with torch.no_grad():
        for xh, xs, d, y in val_loader:
            mu_b, sig_b = model_demo(xh, xs, d)
            vl.append(gaussian_nll(mu_b, sig_b, y).item())
    sched.step()
    tr_nll, va_nll = float(np.mean(tl)), float(np.mean(vl))
    train_hist.append(tr_nll); val_hist.append(va_nll)
    print(f"  Época {epoch:2d}/{DEMO_EPOCHS} | train NLL={tr_nll:.4f} | val NLL={va_nll:.4f}")

# Gráfica de curvas de pérdida
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(train_hist, color=COLOR_GREEN, lw=2, label="Train NLL")
ax.plot(val_hist,   color=COLOR_YELLOW, lw=2, linestyle="--", label="Val NLL")
ax.set_xlabel("Época"); ax.set_ylabel("Gaussian NLL (↓ mejor)")
ax.set_title("Curvas de pérdida — Gaussian Negative Log-Likelihood")
ax.legend()
plt.tight_layout(); plt.show()

print()
print("Nota: NLL negativo significa que la densidad de probabilidad promedio > 1/e")
print("→ el modelo asigna alta probabilidad a las observaciones reales (bien calibrado)")


## 6. Evaluación del modelo entrenado

Cargamos el mejor checkpoint y evaluamos en el conjunto de **test** (parcelas no vistas
durante el entrenamiento).


In [ ]:
# Carga el modelo de producción ya entrenado con 60 épocas
if not MODEL_PATH.exists():
    print("⚠️  Ejecuta primero: make train-likelihood")
else:
    test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=0)
    test_nll = []
    per_idx_mse = np.zeros(5)
    per_idx_mae = np.zeros(5)
    n_samples = 0

    with torch.no_grad():
        for xh, xs, d, y in test_loader:
            mu_b, sig_b = model(xh, xs, d)
            test_nll.append(gaussian_nll(mu_b, sig_b, y).item())
            residuals = (y - mu_b).numpy()
            per_idx_mse += (residuals ** 2).sum(axis=0)
            per_idx_mae += np.abs(residuals).sum(axis=0)
            n_samples   += len(y)

    per_idx_mse = np.sqrt(per_idx_mse / n_samples)   # RMSE
    per_idx_mae = per_idx_mae / n_samples

    print(f"Test NLL (producción):  {np.mean(test_nll):.4f}")
    print(f"Val  NLL (mejor ckpt):  {ckpt['best_val_nll']:.4f}")
    print()
    print(f"{'Índice':8} {'RMSE':10} {'MAE':10}  (espacio normalizado [0,1])")
    print("─" * 36)
    for i, ch in enumerate(CHANNEL_NAMES):
        flag = "★ " if ch == "NDMI" else "  "
        print(f"  {flag}{ch:6} {per_idx_mse[i]:10.5f} {per_idx_mae[i]:10.5f}")


In [ ]:
# Calibración: ¿los sigma predichos son confiables?
# Esperamos que ~68% de los residuos caigan dentro de ±1σ (normal bien calibrada).
if MODEL_PATH.exists():
    residuals_norm = []
    with torch.no_grad():
        for xh, xs, d, y in test_loader:
            mu_b, sig_b = model(xh, xs, d)
            z_b = ((y - mu_b) / sig_b).numpy()   # z-scores de test
            residuals_norm.extend(z_b.flatten())

    residuals_norm = np.array(residuals_norm)
    within_1s = np.mean(np.abs(residuals_norm) <= 1) * 100
    within_2s = np.mean(np.abs(residuals_norm) <= 2) * 100

    print(f"Calibración del modelo:")
    print(f"  Observaciones dentro de ±1σ: {within_1s:.1f}%  (ideal teórico: 68.3%)")
    print(f"  Observaciones dentro de ±2σ: {within_2s:.1f}%  (ideal teórico: 95.4%)")

    # Histograma de z-scores vs normal estándar esperada
    fig, ax = plt.subplots(figsize=(7, 3.5))
    x_norm = np.linspace(-4, 4, 300)
    ax.hist(np.clip(residuals_norm, -4, 4), bins=80, density=True,
            color=COLOR_GREEN, alpha=.55, label="z-scores modelo")
    ax.plot(x_norm, stats.norm.pdf(x_norm), color="#1C2018", lw=2,
            linestyle="--", label="N(0,1) teórica")
    ax.set_xlabel("z-score  (y_obs − μ) / σ")
    ax.set_ylabel("Densidad")
    ax.set_title("Calibración: ¿los σ predichos corresponden a la dispersión real?\n"
                 "Si el histograma se ajusta bien a la curva, el modelo está calibrado")
    ax.legend()
    plt.tight_layout(); plt.show()


## 7. Análisis de una parcela: distribución esperada

Para una parcela concreta, inspeccionamos qué distribución predice el modelo
y dónde cae la última observación dentro de esa distribución.


In [ ]:
PARCEL_ID = "H1"   # cambia aquí para explorar otras parcelas

# Carga serie
npz      = np.load(SIGNALS_DIR / f"{PARCEL_ID}.npz", allow_pickle=True)
data_p   = npz["data"].astype(np.float32)    # (T, 5)
doy_p    = npz["doy"].astype(np.float32)
dates_p  = npz["dates"]

# Prepara entradas
x_hist   = data_p[-(WINDOW_SIZE+1):-1]       # (24, 5)
y_obs    = data_p[-1]                         # (5,) — observación a evaluar
x_static = data_p.mean(axis=0)               # (5,) — huella del terreno
target_doy = int(doy_p[-1])

# Inferencia
if MODEL_PATH.exists():
    with torch.no_grad():
        mu_t, sig_t = model(
            torch.from_numpy(x_hist).unsqueeze(0),
            torch.from_numpy(x_static).unsqueeze(0),
            torch.tensor([float(target_doy)]),
        )
    mu_arr    = mu_t.numpy()[0]
    sigma_arr = sig_t.numpy()[0]
    z_scores  = (y_obs - mu_arr) / sigma_arr

    print(f"Parcela: {PARCEL_ID}  |  Última fecha: {dates_p[-1]}  |  DOY: {target_doy}")
    print()
    print(f"{'Índice':6} {'Observado':12} {'μ (esperado)':14} {'σ':8} {'z-score':10}  Interpretación")
    print("─" * 75)
    for i, ch in enumerate(CHANNEL_NAMES):
        z = z_scores[i]
        flag = "★" if ch == "NDMI" else " "
        interp = ("por encima" if z > 0 else "por debajo") + " de lo esperado"
        print(f"{flag}{ch:5}  {y_obs[i]:12.4f} {mu_arr[i]:14.4f} {sigma_arr[i]:8.4f} {z:+10.3f}  {interp}")


In [ ]:
if MODEL_PATH.exists():
    fig, axes = plt.subplots(1, 5, figsize=(14, 3.8))
    fig.suptitle(
        f"Parcela {PARCEL_ID} — Distribución predicha (μ ± σ) vs. observación real",
        fontsize=12,
    )

    for i, (ch, ax) in enumerate(zip(CHANNEL_NAMES, axes)):
        mu_i, sig_i, obs_i = mu_arr[i], sigma_arr[i], y_obs[i]
        z_i = z_scores[i]

        x_rng = np.linspace(mu_i - 3.8*sig_i, mu_i + 3.8*sig_i, 400)
        y_pdf = stats.norm.pdf(x_rng, mu_i, sig_i)

        ax.plot(x_rng, y_pdf, color=COLOR_GREEN, lw=2)
        ax.fill_between(x_rng, y_pdf, alpha=.12, color=COLOR_GREEN)

        # Sombrear zona fuera de ±2σ
        ax.fill_between(x_rng, y_pdf,
                        where=(x_rng < mu_i - 2*sig_i),
                        alpha=.25, color=COLOR_RED)
        ax.fill_between(x_rng, y_pdf,
                        where=(x_rng > mu_i + 2*sig_i),
                        alpha=.15, color="royalblue")

        # Líneas de referencia
        ax.axvline(mu_i,  color=COLOR_GREEN, lw=1.5, linestyle="--")
        ax.axvline(obs_i, color=COLOR_RED,   lw=2,   linestyle="-",
                   label=f"obs={obs_i:.4f}")

        # Anotación del z-score
        zcolor = COLOR_RED if z_i < -2 else COLOR_YELLOW if z_i < -1 else COLOR_GREEN
        ax.text(0.5, 0.93, f"z = {z_i:+.3f}", ha="center", va="top",
                transform=ax.transAxes, fontsize=9, fontweight="bold", color=zcolor)

        ax.set_title(
            f"{'★ ' if ch == 'NDMI' else ''}{ch}",
            fontsize=10,
            fontweight="bold" if ch == "NDMI" else "normal",
        )
        ax.set_yticks([])
        ax.set_xlabel("valor normalizado", fontsize=8)
        if i == 0:
            ax.legend(fontsize=8, loc="upper left")

    plt.tight_layout()
    plt.show()


## 8. Z-score por índice: señal visual de estrés

La gráfica de barras es la visualización más directa: **barras negativas** significan que
el índice está por debajo de lo que el modelo esperaba dado el terreno + historial + fecha.
Las zonas grises marcan ±1σ y ±2σ.


In [ ]:
if MODEL_PATH.exists():
    bar_colors = [
        COLOR_RED    if z < -2  else
        COLOR_YELLOW if z < -1  else
        COLOR_GREEN  if z >= 0  else
        "#66BB6A"    # verde claro (leve negativo)
        for z in z_scores
    ]

    fig, ax = plt.subplots(figsize=(8, 4.2))
    bars = ax.bar(CHANNEL_NAMES, z_scores, color=bar_colors, alpha=.85,
                  zorder=3, width=.55, edgecolor="white", linewidth=.5)

    # Líneas de referencia ±1σ y ±2σ
    ax.axhline(0,  color="#9E9E9E", linewidth=1.2, zorder=2)
    ax.axhline( 1, color="#BDBDBD", linewidth=1, linestyle="--", label="±1σ", zorder=2)
    ax.axhline(-1, color="#BDBDBD", linewidth=1, linestyle="--", zorder=2)
    ax.axhline( 2, color="#757575", linewidth=1, linestyle=":",  label="±2σ", zorder=2)
    ax.axhline(-2, color="#757575", linewidth=1, linestyle=":",  zorder=2)

    # Zonas de color de fondo
    ax.fill_between([-0.5, 4.5], [-1,-1], [1,1],   alpha=.05, color="grey",   zorder=0)
    ax.fill_between([-0.5, 4.5], [-2,-2], [-1,-1], alpha=.06, color=COLOR_YELLOW, zorder=0)
    ax.fill_between([-0.5, 4.5], [ 1, 1], [ 2, 2], alpha=.04, color=COLOR_GREEN,  zorder=0)

    # Etiquetas de valor sobre las barras
    for bar, z in zip(bars, z_scores):
        va = "bottom" if z >= 0 else "top"
        offset = 0.07 if z >= 0 else -0.09
        ax.text(bar.get_x() + bar.get_width()/2, z + offset,
                f"{z:+.3f}", ha="center", va=va, fontsize=9, fontweight="bold")

    # Leyenda
    patches = [
        mpatches.Patch(color=COLOR_GREEN,  label="z ≥ 0  (por encima de lo esperado)"),
        mpatches.Patch(color=COLOR_YELLOW, label="−2σ < z < −1σ  (estrés moderado)"),
        mpatches.Patch(color=COLOR_RED,    label="z ≤ −2σ  (estrés severo)"),
    ]
    ax.legend(handles=patches + [
        plt.Line2D([0],[0], color="#BDBDBD", linestyle="--", label="±1σ"),
        plt.Line2D([0],[0], color="#757575", linestyle=":",  label="±2σ"),
    ], fontsize=8, loc="lower right")

    ax.set_ylabel("Desviaciones estándar (z-score)", fontsize=10)
    ax.set_title(
        f"Parcela {PARCEL_ID} — z-score por índice respecto a la distribución esperada\n"
        f"(barras negativas = señal de estrés hídrico)",
        fontsize=11,
    )
    ax.set_xlim(-0.5, 4.5)
    plt.tight_layout()
    plt.show()


## 9. Clasificación por CDF multivariada

Una vez que tenemos el z-score de cada índice, los combinamos con pesos (NDMI domina)
y obtenemos un **percentil CDF** que determina la clase de estrés.

Este es exactamente el mecanismo que describió el asesor: la CDF multivariada da los
percentiles de forma fundamentada, no arbitraria.


In [ ]:
if MODEL_PATH.exists():
    # Pesos por índice: NDMI domina por ser el indicador de humedad de hoja más directo
    STRESS_WEIGHTS = np.array([0.15, 0.20, 0.40, 0.15, 0.10])   # NDVI NDWI NDMI NDRE EVI
    assert STRESS_WEIGHTS.sum() == 1.0, "Los pesos deben sumar 1"

    stress_signal = float(np.dot(-z_scores, STRESS_WEIGHTS))
    percentile    = float(stats.norm.cdf(stress_signal) * 100)

    if percentile < 25:   cls, label, color = 0, "Sin estrés",      COLOR_GREEN
    elif percentile < 60: cls, label, color = 1, "Estrés moderado", COLOR_YELLOW
    else:                 cls, label, color = 2, "Estrés severo",   COLOR_RED

    print(f"Parcela {PARCEL_ID}")
    print(f"  Señal de estrés combinada  : {stress_signal:+.4f}")
    print(f"  Percentil CDF  Φ(señal)   : {percentile:.1f}%")
    print(f"  Clasificación              : {label}")
    print()

    # Visualización de la CDF con la posición de esta parcela
    fig, ax = plt.subplots(figsize=(8, 4))
    x_rng   = np.linspace(-3, 3, 500)
    cdf_pct = stats.norm.cdf(x_rng) * 100

    ax.plot(x_rng, cdf_pct, color="#1C2018", lw=2)

    ax.fill_between(x_rng, cdf_pct, where=(cdf_pct < 25),             alpha=.25, color=COLOR_GREEN,  label="Sin estrés (<P25)")
    ax.fill_between(x_rng, cdf_pct, where=((cdf_pct >= 25) & (cdf_pct < 60)), alpha=.25, color=COLOR_YELLOW, label="Moderado (P25–P60)")
    ax.fill_between(x_rng, cdf_pct, where=(cdf_pct >= 60),            alpha=.25, color=COLOR_RED,    label="Severo (≥P60)")

    ax.axvline(stress_signal, color=color, lw=2.5, linestyle="--",
               label=f"{PARCEL_ID}: señal={stress_signal:+.3f} → P{percentile:.0f} → {label}")
    ax.axhline(percentile, color=color, lw=1, linestyle=":")
    ax.plot(stress_signal, percentile, "o", color=color, markersize=9, zorder=5)

    ax.axhline(25, color="#BDBDBD", lw=0.8, linestyle="--")
    ax.axhline(60, color="#BDBDBD", lw=0.8, linestyle="--")

    ax.set_xlabel("Señal de estrés combinada (z ponderado, NDMI × 0.40)", fontsize=10)
    ax.set_ylabel("Percentil CDF · Φ(señal) × 100", fontsize=10)
    ax.set_title(
        "CDF multivariada → clasificación de estrés\n"
        "La posición en la curva determina la clase — sin umbral arbitrario",
        fontsize=11,
    )
    ax.legend(fontsize=9)
    ax.set_ylim(-3, 103)
    plt.tight_layout()
    plt.show()


## 10. Comparación: E3 Stacking vs GP vs Likelihood NN

Para un conjunto de parcelas de test, comparamos la clasificación de estrés que
produce cada experimento.


In [ ]:
import sys
sys.path.insert(0, str(ROOT))
from api.likelihood_predictor import LikelihoodPredictor
from src.models.gp.parcel_gp import load_parcel_series, ParcelGaussianProcess

SAMPLE_PARCELS = split["test"][:8]

if MODEL_PATH.exists():
    lk_pred = LikelihoodPredictor(MODEL_PATH, device="cpu")
    stress_weights_np = np.array([0.15, 0.20, 0.40, 0.15, 0.10])

    rows = []
    for pid in SAMPLE_PARCELS:
        path = SIGNALS_DIR / f"{pid}.npz"
        if not path.exists():
            continue
        npz_i  = np.load(path, allow_pickle=True)
        data_i = npz_i["data"].astype(np.float32)
        doy_i  = npz_i["doy"].astype(np.float32)

        x_h = data_i[-(WINDOW_SIZE+1):-1]
        y_o = data_i[-1]
        x_s = data_i.mean(axis=0)
        doy_last = int(doy_i[-1])

        # Exp E: Likelihood NN
        lk_res  = lk_pred.predict_with_doy(x_h, x_s, y_o, doy_last)
        lk_cls  = lk_res["stress_class"]
        lk_pct  = lk_res["likelihood_percentile"]
        lk_z    = lk_res["combined_z_score"]

        # Exp D: GP individual (NDMI)
        try:
            series_gp = load_parcel_series(pid, "NDMI", SIGNALS_DIR, NORM_PATH)
            gp = ParcelGaussianProcess().fit(series_gp.days, series_gp.values)
            z_gp, lbl_gp = gp.anomaly_score(series_gp.days[-1], series_gp.values[-1])
            gp_cls = {"sin_estres": 0, "moderado": 1, "severo": 2}.get(lbl_gp, -1)
        except Exception:
            z_gp, gp_cls = float("nan"), -1

        # Coincidencia
        match = "✅" if lk_cls == gp_cls else "⚠️"

        rows.append({
            "Parcela":     pid,
            "GP z-score":  f"{z_gp:+.2f}",
            "GP clase":    ["🟢 Sin","🟡 Mod","🔴 Sev","-"][gp_cls],
            "LK % CDF":    f"{lk_pct:.0f}%",
            "LK z comb":   f"{lk_z:+.3f}",
            "LK clase":    ["🟢 Sin","🟡 Mod","🔴 Sev"][lk_cls],
            "Coinciden":   match,
        })

    df_cmp = pd.DataFrame(rows).set_index("Parcela")
    print(df_cmp.to_string())


## 11. Limitaciones (honestidad > impresionar)

1. **Sin los 10 canales raw Sentinel-2** — el asesor mencionó que los 10 canales son
   indispensables para capturar las características visuales del terreno. En este experimento
   usamos los **5 índices derivados** como sustituto; son suficientes para el demo, pero
   un sistema productivo debería acceder a las bandas crudas.

2. **Sin covarianza entre índices** — asumimos diagonal (independencia entre índices).
   Una versión más precisa modelaría la covarianza completa (ej. correlación NDMI–NDWI)
   con una Gaussiana multivariada full-covariance.

3. **Dataset pequeño** — 100 parcelas, ~25k muestras de entrenamiento. El modelo aprende
   patrones de estacionalidad, pero no generaliza fuera del tile Sentinel del experimento.

4. **Etiquetas sin verificación de campo** — igual que en E3 Stacking, no existe groundtruth
   agronómica; la "correctitud" del modelo se evalúa contra NLL, no contra diagnósticos reales.

5. **Los umbrales de CDF (25%/60%) siguen siendo decisiones de diseño** — aunque ahora
   la distribución subyacente está bien fundamentada (Gaussian NLL), los cortes de percentil
   aún requieren validación agronómica con datos de campo.


## 12. Conclusiones

| Aspecto | Resultado |
|---|---|
| Fundamento matemático | ✅ Umbrales emergen de la PDF ajustada por máxima verosimilitud |
| Sensible al terreno | ✅ La huella espectral distingue parcelas con diferente suelo |
| Sensible a la fecha | ✅ Codificación sinusoidal DOY captura estacionalidad |
| Contextual al historial | ✅ Transformer sobre T=24 observaciones anteriores |
| Salida interpretable | ✅ Distribución completa (μ, σ, z-score) por cada índice |
| Integrado al dashboard | ✅ Endpoint `/parcels/{id}/likelihood` + visualización en tab Tendencias |

### Flujo completo del sistema AguaVerde (tres capas)

```
┌─────────────────────────────────────────────────────────────────────┐
│  Capa 1: E3 Stacking (modelo principal, F1-macro=0.8868)           │
│  35 features tabulares → clase de estrés + probabilidades          │
│                                                                     │
│  Capa 2: Experimento D — Gaussian Process por parcela              │
│  Serie temporal 1 índice → z-score contextual al historial propio  │
│                                                                     │
│  Capa 3: Experimento E — Red Neuronal Probabilística (este nb)     │
│  5 índices + terreno + fecha → (μ,σ) → percentil CDF multivariado │
└─────────────────────────────────────────────────────────────────────┘
```

Los tres experimentos son **complementarios**, no excluyentes.
La capa 3 responde directamente a la pregunta del asesor:
> *¿Cómo puedes fundamentar que un índice en ese valor significa estrés?*
> Porque la observación está en el percentil X de la distribución que el modelo espera
> para ese terreno, en esa fecha, dado ese historial — y eso lo aprendió por máxima verosimilitud.
